# Kushmanda: answer from the evidence, cite it, or decline

Each question comes with three passages. A good answer is written from one passage and ends with its citation, e.g. `[2]`; when no passage answers, the right reply is `The context does not answer this.` Below: the data, and the baselines any model must beat.

Kushmanda is one of the nine models of **Sati**, the investigation agent of [Pashupatastra](https://github.com/preetraval45/PASHUPATASTRA), each named for a form of Navadurga. This notebook reproduces its core result on the public dataset, with the split that keeps the test honest.

In [ ]:
import glob, os
def find(pattern):
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    assert hits, f"{pattern} not found under /kaggle/input"
    return hits[0]
import json, re, pandas as pd
def load(name):
    return pd.DataFrame([json.loads(line) for line in open(find(name), encoding="utf-8")])
train, test = load("train.jsonl"), load("test.jsonl")
print(len(train), "train /", len(test), "test")
print(test["kind"].value_counts().to_dict())
r = test.iloc[0]
print("Q:", r["question"])
for i, p in enumerate(r["passages"], 1):
    print(f"[{i}]", p[:160])
print("A:", r["answer"])

In [ ]:
def words(s):
    return set(re.findall(r"[a-z0-9]+", s.lower()))
def pick(q, passages):
    scores = [len(words(q) & words(p)) for p in passages]
    return max(range(len(passages)), key=lambda i: scores[i]) + 1
def cited(a):
    m = re.search(r"\[(\d)\]\s*$", a)
    return int(m.group(1)) if m else None
gold = [cited(a) for a in test["answer"]]
picked = [pick(q, ps) for q, ps in zip(test["question"], test["passages"])]
answerable = [g is not None for g in gold]
print(f"word overlap picks the right passage: {sum(p == g for p, g in zip(picked, gold) if g) / sum(answerable):.3f}")
print(f"share of questions that must be declined: {1 - sum(answerable) / len(test):.3f}")

**Reading it:** picking the right passage is easy for word overlap; writing the answer, and knowing when to decline, is the task. Kushmanda (17M parameters, trained from scratch on this data's train split) reaches ROUGE-L 0.938 on required actions and declines on 0.543 of unanswerable questions, but copies exact identifiers worse than word overlap, so Sati takes identifiers from retrieval and lets the model write around them.